# EXP06 — Controles e integridade dos arquivos

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Conferência de integridade

Liste os arquivos que você deseja comparar no arquivo local `data/processed/manifesto_sha256.csv` com as colunas `arquivo,sha256`. Os caminhos devem ficar dentro da pasta do projeto. Evite incluir arquivos privados no Git.

In [ ]:
from pathlib import Path
import pandas as pd
from ethmix.execucao import abrir_dados, sha256

manifesto = abrir_dados(RAIZ / "data" / "processed" / "manifesto_sha256.csv", ["arquivo", "sha256"])
linhas = []
for item in manifesto.itertuples(index=False):
    caminho = (RAIZ / str(item.arquivo)).resolve()
    if not caminho.is_relative_to(RAIZ.resolve()):
        raise ValueError("Caminho fora da pasta do projeto")
    calculado = sha256(caminho)
    linhas.append({"arquivo": str(item.arquivo), "igual": calculado == str(item.sha256).lower(),
                   "calculado": calculado})
resultado = pd.DataFrame(linhas)
display(resultado)

## 3. Interpretação

Hash igual comprova correspondência de bytes ao manifesto informado. Isso não garante que o dataset foi rotulado corretamente nem que o contrato foi identificado. A verificação da função histórica permanece separada.